# StylePops Bootstrap Pipeline — İP-1 → İP-3

## ⚠️ ÖNEMLİ: `.ipynb` dosyasını bilgisayardan yüklemeyin!

Colab **sadece notebook dosyasını** alır; `data/`, `scripts/` vb. gelmez.
Bu yüzden sol panelde sadece `.ipynb` görürsünüz — bu normal.

**Doğru yol:** Aşağıdaki **"0. GitHub Clone"** hücresini çalıştırın.
Tüm proje `/content/StylePops_Modules/` altına iner.

**Colab'ı şu linkten açın (önerilen):**
https://colab.research.google.com/github/valueisinvalid/StylePops_Modules/blob/main/notebooks/StylePops_Bootstrap_Pipeline.ipynb

---

**Bu notebook:**
1. Bootstrap envanteri yükler (200 parça)
2. İP-1: Özellik vektörü üretir (SBERT + renk + kumaş)
3. İP-2: Termal model (`hedef_Clo`, `efektif_Clo`) + LightGBM v0.1
4. İP-3: MCDM kombin sıralaması (`Rank = Final_Skor − |hedef_Clo − Total_Clo|`)

## 0. GitHub'dan Projeyi Clone Et (İLK ÇALIŞTIRILACAK HÜCRE)

Bu hücreden sonra sol panelde **Files → StylePops_Modules → data** klasörünü görmelisiniz.

In [ ]:
import os

REPO_URL = "https://github.com/valueisinvalid/StylePops_Modules.git"
PROJECT_ROOT = "/content/StylePops_Modules"

if os.path.isdir(PROJECT_ROOT):
    !cd {PROJECT_ROOT} && git pull
else:
    !git clone {REPO_URL} {PROJECT_ROOT}

assert os.path.isdir(PROJECT_ROOT), f"Klasör bulunamadı: {PROJECT_ROOT}"
print("Proje kökü:", PROJECT_ROOT)
!ls {PROJECT_ROOT}

In [ ]:
# Dosyaları doğrula + etiket kontrolü
!head -n 3 {PROJECT_ROOT}/data/bootstrap/combinations_200.csv
!grep -c ",StylePops Bootstrap" {PROJECT_ROOT}/data/bootstrap/combinations_200.csv || echo "0 etiketli satır — git pull gerekebilir"

## 1. Kurulum (pip)

In [ ]:
!pip install -q sentence-transformers lightgbm scikit-learn pandas numpy

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path(PROJECT_ROOT) / "data"
LOOKUPS = DATA / "lookups"
BOOTSTRAP = DATA / "bootstrap"

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

fabric_props = load_json(LOOKUPS / "fabric_properties.json")
target_clo = load_json(LOOKUPS / "target_clo_points.json")
coverage_ratios = load_json(LOOKUPS / "coverage_ratios.json")
season_templates = load_json(LOOKUPS / "season_templates.json")

garments_payload = load_json(BOOTSTRAP / "garments_200.json")
garments = {g["id"]: g for g in garments_payload["garments"]}
combos_df = pd.read_csv(BOOTSTRAP / "combinations_200.csv")
combos_df["aesthetic_score"] = pd.to_numeric(combos_df["aesthetic_score"], errors="coerce")
combos_df["thermal_score"] = pd.to_numeric(combos_df["thermal_score"], errors="coerce")
n_labeled = combos_df["aesthetic_score"].notna().sum()
print(f"Etiketli kombin (CSV yükleme): {n_labeled} / {len(combos_df)}")
if n_labeled == 0:
    print("⚠️ Etiket yok! Önce '0. GitHub Clone' hücresini tekrar çalıştırın (git pull), sonra bu hücreyi yeniden çalıştırın.")

MATERIALS = fabric_props["materials"]
THERMAL_CATS = fabric_props["thermal_categories"]
CLO_POINTS = target_clo["target_clo_points"]
WEATHER_SCENARIOS = target_clo["weather_scenarios"]

COVERAGE_DEFAULTS = coverage_ratios["coverage_by_subcategory"]

import sys
sys.path.insert(0, str(Path(PROJECT_ROOT) / "scripts"))
from stylepops_core import (
    effective_clo, effective_ret, ensemble_total_clo,
    apparent_temperature, interpolate_hedef_clo,
    generate_layered_candidates, score_combination as core_score_combination,
)

def garment_clo(g):
    return effective_clo(g, THERMAL_CATS, COVERAGE_DEFAULTS)

def garment_ret(g):
    return effective_ret(g, THERMAL_CATS, COVERAGE_DEFAULTS)

print(f"Parça sayısı: {len(garments)}")
print(f"Kombin sayısı: {len(combos_df)}")
g = list(garments.values())[0]
print(f"Örnek: {g['name']} ({g.get('layer_role')}) -> Clo={garment_clo(g)}")
combos_df.head(3)

## 2. İP-1 — Veri İşleme ve Özellik Vektörü

In [ ]:
# Clo: thermal_category (giysi düzeyi) × kaplama normalizasyonu
# Etek (0.60) vs pantolon (0.90) farkı coverage_ratio ile ölçeklenir
for label, sub in [("Jean", "jeans"), ("Midi etek", "skirt")]:
    sample = next(g for g in garments.values() if g["subcategory"] == sub)
    print(f"{label}: coverage={sample['coverage_ratio']} -> Clo={garment_clo(sample)}")

In [ ]:
from sentence_transformers import SentenceTransformer

# GPU varsa otomatik kullanır
sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

descriptions = [g["description"] for g in garments.values()]
ids = list(garments.keys())
style_embeddings = sbert.encode(descriptions, show_progress_bar=True, batch_size=32)
style_vectors = {gid: emb for gid, emb in zip(ids, style_embeddings)}

print("SBERT boyutu:", style_embeddings.shape)

In [ ]:
def garment_feature_vector(garment):
    """771 boyutlu özellik vektörünün bootstrap yaklaşımı."""
    style = style_vectors[garment["id"]]
    lab = garment["color_lab"]
    color = np.array([lab["L"], lab["a"], lab["b"]])
    thermal = np.array([
        garment_clo(garment),
        garment_ret(garment),
        garment["coverage_ratio"],
    ])
    # Kumaş one-hot (14 malzeme)
    mat_names = list(MATERIALS.keys())
    mat_vec = np.zeros(len(mat_names))
    for f in garment["fabric_composition"]:
        if f["material"] in mat_names:
            mat_vec[mat_names.index(f["material"])] = f["pct"] / 100
    return np.concatenate([style, color, thermal, mat_vec])


def combo_feature_vector(piece_ids):
    vecs = [garment_feature_vector(garments[pid]) for pid in piece_ids if pid]
    return np.mean(vecs, axis=0)

sample = combo_feature_vector(["G001", "G079", "G040"])
print("Kombin özellik boyutu:", sample.shape)

## 3. İP-2 — Termal Modelleme

In [ ]:
def weather_to_hedef_clo(T_hava, RH, V_ruzgar):
    T_hissedilen = apparent_temperature(T_hava, RH, V_ruzgar)
    return T_hissedilen, interpolate_hedef_clo(T_hissedilen, CLO_POINTS)

# Test: kış senaryosu
T_h, hedef = weather_to_hedef_clo(2, 65, 25)
print(f"T_hissedilen={T_h}°C -> hedef_Clo={hedef}")

## 4. İP-2 — Uyarlayıcı Puanlama (LightGBM v0.1)

Etiketler boşsa kural tabanlı fallback devreye girer (Risk 1/4 B Planı).

In [ ]:
def rule_based_aesthetic_score(piece_ids):
    """Risk 4 B Planı — renk uyumu + temel parça bonusları."""
    pieces = [garments[pid] for pid in piece_ids if pid]
    if not pieces:
        return 2.5
    labs = np.array([[p["color_lab"]["L"], p["color_lab"]["a"], p["color_lab"]["b"]] for p in pieces])
    color_std = np.std(labs, axis=0).mean()
    color_score = max(1.0, 5.0 - color_std / 8)
    categories = {p["category"] for p in pieces}
    bonus = 0.3 if "top" in categories and "bottom" in categories else 0
    return min(5.0, color_score + bonus)


def build_training_data(combos_df):
    df = combos_df.copy()
    df["aesthetic_score"] = pd.to_numeric(df["aesthetic_score"], errors="coerce")
    df["thermal_score"] = pd.to_numeric(df["thermal_score"], errors="coerce")
    labeled = df.dropna(subset=["aesthetic_score"])
    print(f"LightGBM eğitim seti: {len(labeled)} etiketli kombin")
    X, y_est, y_th = [], [], []
    for _, row in labeled.iterrows():
        piece_ids = [p for p in str(row["piece_ids"]).split("|") if p]
        X.append(combo_feature_vector(piece_ids))
        y_est.append(float(row["aesthetic_score"]))
        if pd.notna(row["thermal_score"]):
            y_th.append(float(row["thermal_score"]))
    return np.array(X), np.array(y_est), np.array(y_th) if y_th else None


X_train, y_est_train, y_th_train = build_training_data(combos_df)
lgbm_aesthetic = None
USE_LGBM = len(y_est_train) >= 20  # en az 20 etiketli kombin

if USE_LGBM:
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score

    X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_est_train, test_size=0.2, random_state=42)
    lgbm_aesthetic = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
    lgbm_aesthetic.fit(X_tr, y_tr)
    r2 = r2_score(y_te, lgbm_aesthetic.predict(X_te))
    print(f"LightGBM estetik R² = {r2:.3f}")
    if r2 < 0.70:
        print("R² < 0.70 — kural tabanlı fallback kullanılacak.")
        lgbm_aesthetic = None
else:
    print(f"Etiketli kombin: {len(y_est_train)} — LightGBM atlandı, kural tabanlı skorlama aktif.")
    print("Çözüm: (1) Clone hücresini çalıştır → git pull  (2) Veri yükleme hücresini yeniden çalıştır  (3) SBERT hücresinden sonra bu hücreyi çalıştır")

In [ ]:
def predict_aesthetic(piece_ids):
    if lgbm_aesthetic is not None:
        return float(lgbm_aesthetic.predict(combo_feature_vector(piece_ids).reshape(1, -1))[0])
    return rule_based_aesthetic_score(piece_ids)

## 5. İP-3 — Çok Kriterli Optimizasyon (MCDM)

In [ ]:
def score_combination(piece_ids, hedef_clo, V_ruzgar=0):
    return core_score_combination(
        piece_ids, garments, hedef_clo, V_ruzgar,
        THERMAL_CATS, COVERAGE_DEFAULTS, predict_aesthetic,
    )


def recommend_outfit(scenario_key, season=None, top_k=5):
    scenario = WEATHER_SCENARIOS[scenario_key]
    T_hissedilen, hedef_clo = weather_to_hedef_clo(
        scenario["T_hava"], scenario["RH_nem"], scenario["V_ruzgar"]
    )
    season = season or scenario.get("season")
    candidates = generate_layered_candidates(
        garments, n_candidates=400, season=season,
        hedef_clo=hedef_clo, V_ruzgar=scenario["V_ruzgar"],
    )
    scored = [score_combination(c, hedef_clo, scenario["V_ruzgar"]) for c in candidates]
    ranked = sorted(scored, key=lambda x: x["rank"], reverse=True)[:top_k]

    print(f"Senaryo: {scenario.get('label_tr', scenario_key)}")
    print(f"T_hissedilen={T_hissedilen}°C | hedef_Clo={hedef_clo}")
    print("-" * 60)
    for i, r in enumerate(ranked, 1):
        names = " + ".join(garments[pid]["name"] for pid in r["piece_ids"])
        print(f"#{i} Rank={r['rank']} | {len(r['piece_ids'])} parça")
        print(f"   {names}")
        print(f"   katmanlar: {r['layers']}")
        print(f"   estetik={r['skor_estetik']} | Clo={r['total_Clo_C']} | ΔClo={r['delta_Clo']}")
        if r.get("accessory_hints"):
            print(f"   eksik öneri: {r['accessory_hints']}")
    return ranked

## 6. Demo — 4 Mevsim Senaryosu

In [ ]:
for scenario in ["kis_soguk_ruzgarli", "sonbahar_serin", "ilkbahar_ilik", "yaz_sicak"]:
    recommend_outfit(scenario, top_k=3)
    print()

## 7. Bootstrap Kombinleri Değerlendir (Etiketleme Sonrası)

In [ ]:
def evaluate_labeled_combos(combos_df):
    results = []
    for _, row in combos_df.iterrows():
        piece_ids = row["piece_ids"].split("|")
        hedef = row["hedef_Clo"]
        V = row.get("V_ruzgar", 0)
        scored = score_combination(piece_ids, hedef, V)
        scored["combo_id"] = row["combo_id"]
        scored["season"] = row["season"]
        if not pd.isna(row.get("aesthetic_score")) and row.get("aesthetic_score") != "":
            scored["label_aesthetic"] = float(row["aesthetic_score"])
        results.append(scored)
    return pd.DataFrame(results)

eval_df = evaluate_labeled_combos(combos_df)
eval_df.groupby("season")[["rank", "delta_Clo"]].mean().round(3)

## 8. Sonuçları Kaydet

In [ ]:
OUT = Path(PROJECT_ROOT) / "outputs"
OUT.mkdir(exist_ok=True)

eval_df.to_csv(OUT / "bootstrap_eval_results.csv", index=False)
print(f"Kaydedildi: {OUT / 'bootstrap_eval_results.csv'}")

if lgbm_aesthetic is not None:
    import joblib
    joblib.dump(lgbm_aesthetic, OUT / "lgbm_aesthetic_v0.1.joblib")
    print(f"Model kaydedildi: {OUT / 'lgbm_aesthetic_v0.1.joblib'}")